# **🏠 Heritage Housing — Data Cleaning**

---

## 📋 1. Data Cleaning Summary

### Objectives

The objective of this notebook is to inspect, clean, and prepare the Heritage Housing dataset for further analysis and modelling.

The notebook will:

* Explore the structure and quality of the dataset.
* Identify and investigate missing values.
* Apply appropriate treatments to categorical and numerical missing data.
* Perform data quality checks.
* Save the cleaned dataset for use in subsequent notebooks.

### Input

* `outputs/datasets/collection/house_prices_records.csv`

### Output

* `outputs/datasets/cleaned/house_prices_cleaned.csv`

---

## ⚙️ 2. Set Up the Notebook

## Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/milestone-project5-heritage-housing-issues/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/workspaces/milestone-project5-heritage-housing-issues'

---

## 📥 3. Load Data

Import heritage housing historical data.

In [4]:
import pandas as pd

df = pd.read_csv("outputs/datasets/collection/house_prices_records.csv")

df.head(3)

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,NaN,460,RFn,...,80.0,0.0,0,8,6,1262,NaN,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,NaN,2001,2002,223500


---

## 🔍 4. Data Exploration

### 4.1 Pandas Profiling Report
Generate an initial overview of the dataset using Pandas Profiling.

In [5]:
import ipywidgets as widgets
from IPython.display import display, clear_output

button = widgets.Button(description="Show report")
output = widgets.Output()

def toggle_report(b):
    with output:
        if button.description == "Show report":
            pandas_report.to_notebook_iframe()
            button.description = "Hide report"
        else:
            clear_output()
            button.description = "Show report"

button.on_click(toggle_report)

display(button, output)

Button(description='Show report', style=ButtonStyle())

Output()

### 4.2 Interpretation of Pandas Profiling Report
* **24 variables** in total
  * **20 numerical variables**
  * **4 categorical variables**
    *  `BsmtExposure` – Basement exposure
    *  `BsmtFinType1` – Type of basement finish
    *  `GarageFinish` – Garage finish level
    *  `KitchenQual` – Kitchen quality
* `OverallQual` and `OverallCond` are numerical variables with an **ordinal interpretation**.
* `BedroomAbvGr` is also a numerical variable, but it represents a **count** rather than a continuous measurement.
* There are **missing values in several variables**.

### 4.3 Missing Values Overview
- First, I checked the number of missing values in each feature and found that the missing values are concentrated in:

* `EnclosedPorch`
* `WoodDeckSF`
* `LotFrontage`
* `GarageFinish`
* `BsmtFinType1`
* `BedroomAbvGr`
* `2ndFlrSF`
* `GarageYrBlt`
* `BsmtExposure`
* `MasVnrArea`


In [6]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing

EnclosedPorch    1324
WoodDeckSF       1305
LotFrontage       259
GarageFinish      235
BsmtFinType1      145
BedroomAbvGr       99
2ndFlrSF           86
GarageYrBlt        81
BsmtExposure       38
MasVnrArea          8
dtype: int64

## 🧹 5. Data Cleaning

### 5.1 Handle Missing Categorical Values

Of the 4 categorical variables, 3 contain missing values:

* GarageFinish — 235 missing values
* BsmtFinType1 — 145 missing values
* BsmtExposure — 38 missing values

The remaining categorical variable, `KitchenQual`, contains **no missing values** and requires no treatment.

First, I will check whether the missing values in the three categorical variables represent the absence of the corresponding feature:

* **GarageFinish** — compare with `GarageArea`.
* **BsmtFinType1** — compare with `TotalBsmtSF`.
* **BsmtExposure** — compare with `TotalBsmtSF`.

A value of `0` in the related numerical variable indicates that the garage or basement is absent.

In [9]:
# Check whether missing categorical values represent absence of the feature

feature_checks = {
    'GarageFinish': 'GarageArea',
    'BsmtFinType1': 'TotalBsmtSF',
    'BsmtExposure': 'TotalBsmtSF'
}

for cat_col, num_col in feature_checks.items():
    missing = df.loc[df[cat_col].isna(), num_col]

    print(f"\n{cat_col} ({len(missing)} missing)")
    print(f"{num_col} = 0: {(missing == 0).sum()}")
    print(f"{num_col} > 0: {(missing > 0).sum()}")


GarageFinish (235 missing)
GarageArea = 0: 81
GarageArea > 0: 154

BsmtFinType1 (145 missing)
TotalBsmtSF = 0: 37
TotalBsmtSF > 0: 108

BsmtExposure (38 missing)
TotalBsmtSF = 0: 37
TotalBsmtSF > 0: 1


The results show that:

* **GarageFinish:** 81 missing values represent no garage, while 154 are genuinely missing.
* **BsmtFinType1:** 37 represent no basement, while 108 are genuinely missing.
* **BsmtExposure:** 37 represent no basement, with only 1 genuinely missing.

All genuinely missing values in these three categories will be replaced with "None"

In [10]:
# Replace NaN with 'None' where the garage/basement is absent

df.loc[
    df['GarageFinish'].isna() & (df['GarageArea'] == 0),
    'GarageFinish'
] = 'None'

df.loc[
    df['BsmtFinType1'].isna() & (df['TotalBsmtSF'] == 0),
    'BsmtFinType1'
] = 'None'

df.loc[
    df['BsmtExposure'].isna() & (df['TotalBsmtSF'] == 0),
    'BsmtExposure'
] = 'None'

In [12]:
# Check what remains after replacing genuinely missing values with none
df[['GarageFinish', 'BsmtFinType1', 'BsmtExposure']].isna().sum()

GarageFinish    154
BsmtFinType1    108
BsmtExposure      1
dtype: int64

I would guess that overall quality of the house would be a good determination of the Garage Quality. I will run the below code to investigate this. 

In [14]:
pd.crosstab(
    df['OverallQual'],
    df['GarageFinish'],
    normalize='index'
).round(2)

GarageFinish,Fin,None,RFn,Unf
OverallQual,,,,
1,0.00,1.00,0.00,0.00
2,0.00,0.50,0.00,0.50
3,0.00,0.30,0.05,0.65
4,0.05,0.24,0.08,0.63
5,0.09,0.09,0.15,0.67
6,0.22,0.04,0.31,0.44
7,0.32,0.00,0.46,0.22
8,0.47,0.01,0.45,0.08
9,0.85,0.00,0.13,0.03


The overall quality of the house is a good measure to use to predict the garagefinish. The higher the overall quality of the house the better the garage finish.

In [15]:
# Find the most common GarageFinish for each OverallQual
garage_finish_by_quality = (
    df[df['GarageFinish'].notna() & (df['GarageFinish'] != 'None')]
    .groupby('OverallQual')['GarageFinish']
    .agg(lambda x: x.mode()[0])
)

garage_finish_by_quality

OverallQual
2     Unf
3     Unf
4     Unf
5     Unf
6     Unf
7     RFn
8     Fin
9     Fin
10    Fin
Name: GarageFinish, dtype: object

In [16]:
# Fill missing GarageFinish based on the most common
# garage finish for each OverallQual

mask = df['GarageFinish'].isna() & (df['GarageArea'] > 0)

df.loc[mask, 'GarageFinish'] = (
    df.loc[mask, 'OverallQual']
    .map(garage_finish_by_quality)
)

In [17]:
df['GarageFinish'].isna().sum()

0

### 5.2 Handle Missing Numerical Values

Of the 20 numerical variables, 7 contain missing values:

* EnclosedPorch — 1,324 missing values
* WoodDeckSF — 1,305 missing values
* LotFrontage — 259 missing values
* BedroomAbvGr — 99 missing values
* 2ndFlrSF — 86 missing values
* GarageYrBlt — 81 missing values
* MasVnrArea — 8 missing values

The remaining 13 numerical variables contain no missing values and require no treatment.

* **Replace with `0`:** `EnclosedPorch`, `WoodDeckSF`, `2ndFlrSF`, and `MasVnrArea`, where missing values can reasonably represent the absence of the feature.
* **Replace with the median:** `LotFrontage`, `BedroomAbvGr`, and `GarageYrBlt`, where `0` would not be a realistic replacement.

The median is preferred for genuinely missing numerical values because it is less sensitive to extreme values than the mean.

In [9]:
# Fill missing values where NaN represents the absence of a feature
absence_cols = [
    'EnclosedPorch',
    'WoodDeckSF',
    '2ndFlrSF',
    'MasVnrArea'
]

df[absence_cols] = df[absence_cols].fillna(0)


# Fill genuinely missing numerical values with the median
median_cols = [
    'LotFrontage',
    'BedroomAbvGr',
    'GarageYrBlt'
]

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

### 5.3 Confirm Missing Values are Resolved

In [12]:
df.isnull().sum().sort_values(ascending=False)

1stFlrSF         0
2ndFlrSF         0
BedroomAbvGr     0
BsmtExposure     0
BsmtFinSF1       0
BsmtFinType1     0
BsmtUnfSF        0
EnclosedPorch    0
GarageArea       0
GarageFinish     0
GarageYrBlt      0
GrLivArea        0
KitchenQual      0
LotArea          0
LotFrontage      0
MasVnrArea       0
OpenPorchSF      0
OverallCond      0
OverallQual      0
TotalBsmtSF      0
WoodDeckSF       0
YearBuilt        0
YearRemodAdd     0
SalePrice        0
dtype: int64

All variables now report zero missing values so our cleanup has been successful!

---

## 🔎 6. Data Quality Checks

Before saving the cleaned dataset, perform final quality checks.

### 6.1 Check for Duplicate Records
No suplicate records have been found which is what we hoped for.

In [10]:
df.duplicated().sum()

0

### 6.2 Check Data Types

This confirms that numerical and categorical variables retain appropriate data types after cleaning.

In [11]:
df.dtypes

1stFlrSF           int64
2ndFlrSF         float64
BedroomAbvGr     float64
BsmtExposure      object
BsmtFinSF1         int64
BsmtFinType1      object
BsmtUnfSF          int64
EnclosedPorch    float64
GarageArea         int64
GarageFinish      object
GarageYrBlt      float64
GrLivArea          int64
KitchenQual       object
LotArea            int64
LotFrontage      float64
MasVnrArea       float64
OpenPorchSF        int64
OverallCond        int64
OverallQual        int64
TotalBsmtSF        int64
WoodDeckSF       float64
YearBuilt          int64
YearRemodAdd       int64
SalePrice          int64
dtype: object

### 6.3 Check Dataset Shape

The number of rows has remained unchanged which is what we would have expected.

In [12]:
df.shape

(1460, 24)

### 6.4 Final Dataset Preview

The dataset is now cleaned while retaining its original categorical variables. Encoding and other feature transformations will be performed later when required for analysis or modelling.

In [13]:
df.head(3)

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,0.0,460,RFn,...,80.0,0.0,0,8,6,1262,0.0,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,0.0,2001,2002,223500


---

## 💾 6. 

---

## ✅ 7. Conclusions